# 02. Process Data
Weather processing, merge, feature engineering.

In [ ]:
import pandas as pd
import numpy as np
import os
import gc
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import BallTree
import pyarrow.dataset as ds
import notebook_const

from src import utils
from src import const
from src.data_process.split import Split

# load data

In [ ]:
base_df = utils.load_data()

In [ ]:
per99 = base_df['arrival_delay'].quantile(0.99)
per1 = base_df['arrival_delay'].quantile(0.01)

REGION_ID_CATEGORIES = sorted(base_df['region_id'].fillna('unknown').astype(str).unique())
REGION_DUMMY_COLUMNS = [f"region_id_{region}" for region in REGION_ID_CATEGORIES]

del base_df

In [ ]:
# Prepare Trip Data (Aggregation)
def aggregate_data(df):
    data = df.copy()

    data['trip_key'] = (
        data['start_date'].astype(str) + '_' +
        data['route_id'].astype(str) + '_' +
        data['direction_id'].astype(str) + '_' +
        data['trip_id'].astype(str)
    )

    data['route_direction_key'] = (
        data['route_id'].astype(str) + '_' +
        data['direction_id'].astype(str)
    )

    seq_group = data.groupby('trip_key')['stop_sequence']
    data['seq_min'] = seq_group.transform('min')
    data['seq_max'] = seq_group.transform('max')

    data['stop_type'] = 'middle'
    data.loc[data['stop_sequence'] == data['seq_min'], 'stop_type'] = 'first'
    data.loc[data['stop_sequence'] == data['seq_max'], 'stop_type'] = 'last'

    group_cols = ['trip_key', 'stop_sequence']
    grouped_arrival = data.groupby(group_cols)['arrival_delay']
    arrival_first = grouped_arrival.transform('first')
    arrival_max = grouped_arrival.transform('max')
    arrival_min = grouped_arrival.transform('min')
    data['arrival_delay'] = np.where(
        data['stop_type'] == 'first',
        arrival_max,
        np.where(data['stop_type'] == 'last', arrival_min, arrival_first)
    ).astype('float32')

    data_unique = (
        data.sort_values(['trip_key', 'stop_sequence'])
        .drop_duplicates(subset=group_cols, keep='first')
        .drop(columns=['seq_min', 'seq_max', 'stop_type'])
    )

    data_unique['arrival_delay'] = data_unique.groupby('trip_key')['arrival_delay'].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )

    return data_unique

In [ ]:
# Load Pre-calculated Static Stop Features
def load_static_stop_features():
    print("Loading Static Stop Features...")
    input_path = f"{const.PROCESSED_DATA_DIR}/static_stop_features.csv"    
    static_features = pd.read_csv(input_path)
    static_features['stop_id'] = static_features['stop_id'].astype(str)
    return static_features

# Execute once
static_stop_features = load_static_stop_features()
print("Columns:", static_stop_features.columns.tolist())

In [ ]:
def process_features(df_process):
    """Apply feature engineering"""
    mask_9956 = df_process['stop_id'].astype(str) == '9956'
    df_process.loc[mask_9956, 'region_id'] = 'maple_ridge'
    df_process['start_date'] = df_process['start_date'].astype(str)
    df_process['region_id'] = df_process['region_id'].fillna('unknown').astype(str)
    df_process['stop_id'] = df_process['stop_id'].astype(str)  # Ensure string for merge

    # Consistent region_id one-hot vectors across all processed files
    expected_region_cols = globals().get('REGION_DUMMY_COLUMNS')
    region_dummies = pd.get_dummies(df_process['region_id'], prefix='region_id')
    if expected_region_cols:
        region_dummies = region_dummies.reindex(columns=expected_region_cols, fill_value=0)
    else:
        region_dummies = region_dummies.reindex(sorted(region_dummies.columns), fill_value=0)
    region_dummies = region_dummies.astype('int8')
    df_process = pd.concat([df_process, region_dummies], axis=1)

    scheduled_time = pd.to_datetime(df_process['scheduled_arrival_time'], utc=True)
    df_process['time_of_day'] = scheduled_time.dt.hour + scheduled_time.dt.minute / 60
    df_process['hour'] = scheduled_time.dt.hour
    df_process['time_sin'] = np.sin(2 * np.pi * df_process['time_of_day'] / 24)
    df_process['time_cos'] = np.cos(2 * np.pi * df_process['time_of_day'] / 24)
    df_process['day_of_week'] = pd.to_datetime(df_process['start_date'], format='%Y%m%d').dt.dayofweek
    df_process['is_weekend'] = (df_process['day_of_week'] >= 6).astype(int)

    # v2 features
    df_process['is_morning_rush_hour'] = ((df_process['hour'] >= 7) & (df_process['hour'] <= 7)).astype(int)
    df_process['is_evening_rush_hour'] = ((df_process['hour'] >= 16) & (df_process['hour'] <= 19)).astype(int)

    df_process['has_detour'] = (df_process['alert_effect_detour'] > 0).astype(int)

    df_process['has_police_alert'] = (df_process['alert_police_activity'] > 0).astype(int)

    # 3. Merge Static Stop Features (Infrastructure & Environment)
    df_process = df_process.merge(static_stop_features, on='stop_id', how='left')
    # Fill NaNs for stops that might not have been in the static set (should be rare)
    cols_to_fill = [c for c in static_stop_features.columns if c != 'stop_id']
    df_process[cols_to_fill] = df_process[cols_to_fill].fillna(0)

    rd_encoder = LabelEncoder()
    df_process['route_direction_encoded'] = rd_encoder.fit_transform(df_process['route_direction_key'])

    # Lag features based on the previous five trips within the same route-direction and stop
    trip_features = df_process[['route_direction_key', 'stop_id', 'trip_key']].copy()
    trip_features['trip_start_time'] = scheduled_time
    trip_features['trip_delay'] = df_process['arrival_delay'].astype('float32')
    trip_features = (
        trip_features.groupby(['route_direction_key', 'stop_id', 'trip_key'], as_index=False)
        .agg(
            trip_start_time=('trip_start_time', 'min'),
            trip_delay=('trip_delay', 'mean')
        )
        .sort_values(['route_direction_key', 'stop_id', 'trip_start_time', 'trip_key'])
    )

    lag_columns = []
    for lag in range(1, 6):
        col = f'lag_arrival_delay_{lag}'
        trip_features[col] = (
            trip_features.groupby(['route_direction_key', 'stop_id'])['trip_delay']
            .shift(lag)
            .astype('float32')
        )
        lag_columns.append(col)

    df_process = df_process.merge(
        trip_features[['route_direction_key', 'stop_id', 'trip_key'] + lag_columns],
        on=['route_direction_key', 'stop_id', 'trip_key'],
        how='left'
    )

    df_process[lag_columns] = df_process[lag_columns].fillna(0.0)
    
    return df_process

In [ ]:
# Select columns including Feature Store features
region_feature_columns = (globals().get('REGION_DUMMY_COLUMNS', []) or [])

# Infrastructure & Environment columns
# Extract dynamically from the static dataframe
infra_env_columns = [c for c in static_stop_features.columns if c != 'stop_id']

base_feature_columns = [
    'trip_key', 'route_direction_key', 'start_date', 'stop_sequence', 'stop_id', 'region_id',
    'scheduled_arrival_time', 'time_bucket', 'hour_of_day', 'day_of_week',
    'time_of_day', 'hour', 'time_sin', 'time_cos',
    'is_weekend', 'is_morning_rush_hour', 'is_evening_rush_hour',
    'arrival_delay',
    'has_active_alert', 'has_detour', 'has_police_alert',
    'route_direction_encoded'
 ]
lag_feature_columns = [
    'lag_arrival_delay_1', 'lag_arrival_delay_2', 'lag_arrival_delay_3',
    'lag_arrival_delay_4', 'lag_arrival_delay_5'
 ]
features = base_feature_columns + region_feature_columns + infra_env_columns + lag_feature_columns

print(f"Total features selected: {len(features)}")
print(f"Added Infrastructure/Environment features: {infra_env_columns}")

base_path = const.GTFS_REALTIME_DIR
output_dir = const.PROCESSED_DATA_DIR
arrival_delay_filter = (ds.field('arrival_delay') >= per1) & (ds.field('arrival_delay') <= per99)
for file in os.listdir(base_path):
    if file.endswith('.parquet'):
        print(f"Processing Data: {file}")
        file_path = os.path.join(base_path, file)
        base_df = pd.read_parquet(file_path)
        
        df_process = aggregate_data(base_df)
        del base_df
        df_process = process_features(df_process)
        
        parquet_file = f'{output_dir}/{file}_processed.parquet'
        df_process[features].to_parquet(parquet_file, index=False, compression='snappy')
        print(f"Saved Processed Data: {parquet_file}")
        del df_process
        gc.collect()

In [ ]:
import pandas as pd
import os
import notebook_const
from src import const, utils


test_df = pd.read_parquet(os.path.join(const.PROCESSED_DATA_DIR, 'analyze_delay_base_20251118_20251124.parquet_processed.parquet'))
test_df.head()

# Train / Test Split

To ensuring a "completely fixed" evaluation dataset, we split the time-series data chronologically and save it.
- Training period: 70%
- Test period: 30%

All models (03 to 06) will perform evaluation using this identical file.

In [ ]:
import notebook_const
from src import const, utils
from src.data_process.split import Split
df_process_selected = utils.load_data(const.PROCESSED_DATA_DIR)

splitter = Split(train_size=0.7)
splitter.split_time_series(df_process_selected)
splitter.print_split_info()
splitter.save_split_data()

del splitter

In [ ]:
# Load the split data we just saved to ensure consistency
df_train, df_test, df_process, split_info = utils.load_split_data_with_combined()

# Create Sequences using shared function (lag-aware)
lag_columns = sorted([col for col in df_process.columns if col.startswith('lag_arrival_delay_')])
requested_past_trips = 5
n_past_trips = min(requested_past_trips, len(lag_columns))
data = utils.prepare_model_data(df_train, df_test, df_process, n_past_trips=n_past_trips)

# Extract variables
X_delays_train, X_features_train, X_agg_train, y_train = \
    data['X_delays_train'], data['X_features_train'], data['X_agg_train'], data['y_train']
X_delays_test, X_features_test, X_agg_test, y_test = \
    data['X_delays_test'], data['X_features_test'], data['X_agg_test'], data['y_test']
n_stops = data['n_stops']

metadata = {
    'n_past_trips': int(n_past_trips),
    'n_stops': int(data['n_stops']),
    'lag_columns': lag_columns[:n_past_trips]
}

utils.save_model_input(
    X_delays_train, X_features_train, X_agg_train, y_train,
    X_delays_test, X_features_test, X_agg_test, y_test,
    metadata
)

In [ ]:
df_train.head()